# Capstone: Find a Circuit

**Pick a behavior. Find the circuit. Write it up.**

You've learned the techniques. Now use them.

**Your mission:** Pick a behavior in GPT-2 small, find the circuit responsible for it, and write up your findings. This is how real interpretability research works — you start with a behavior, form hypotheses, and use the tools from this guide to test them.

This notebook gives you a scaffold, not answers. By the end you should have:
1. A clearly defined behavior with a metric
2. A set of components (heads, MLPs) that implement it
3. Evidence for *how* those components work together
4. An honest assessment of what you don't understand

## Step 0: Choose Your Behavior

Here are some promising behaviors to investigate in GPT-2 small. Pick one, or find your own.

| Behavior | Example Prompt | What to Measure |
|----------|---------------|------------------|
| **Gendered pronouns** | "The nurse said that she/he" | Logit difference: she vs he after gendered occupation |
| **Comparative reasoning** | "Alice is taller than Bob. The tallest person is" | Logit: Alice vs Bob |
| **Acronym completion** | "NASA stands for National Aeronautics and Space" | Logit of "Administration" |
| **Country-capital** | "The capital of France is" | Logit of "Paris" |
| **Rhyming** | "cat, bat, hat, mat, sat, r" | Logit of "rat" (note: "r" + "at" may tokenize differently from "rat" -- verify with your tokenizer) |
| **Negation** | "The cat is not" vs "The cat is" | Distribution shift |
| **List continuation** | "red, blue, green," | Logit of color words vs non-colors |

**What makes a good behavior to study:**
- Clear correct answer (so you can define a metric)
- Works reliably on GPT-2 small (test first!)
- Not already fully explained (IOI is done — find something new)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small")
model.eval()

# Step 0: Define your behavior and test it
# Fill these in:
BEHAVIOR_NAME = "your_behavior_here"
clean_prompt = "..."  # prompt where the model exhibits the behavior
expected_token = "..."  # the token the model should produce

# Test that the model actually exhibits this behavior
with torch.no_grad():
    logits = model(clean_prompt)
    probs = logits[0, -1].softmax(dim=-1)
    token_id = model.to_single_token(expected_token)
    print(f"Behavior: {BEHAVIOR_NAME}")
    print(f"Prompt: {clean_prompt}")
    print(f"P({expected_token}) = {probs[token_id].item():.4f}")
    print(f"Rank of '{expected_token}': {(probs > probs[token_id]).sum().item()}")

    # Also check: what does the model actually predict?
    top5 = probs.topk(5)
    for i in range(5):
        tok = model.to_string([top5.indices[i].item()])
        print(f"  Top {i+1}: {tok!r} ({top5.values[i].item():.4f})")

## Step 1: Define Your Metric

A metric converts model behavior into a single number. Good metrics:
- Are positive when the model does the right thing
- Are larger when the model is more confident
- Work for comparing clean vs patched runs

The standard choice is **logit difference**: `logit(correct) - logit(incorrect)`. For behaviors without a clear "incorrect" token, use `logit(correct) - mean(logits)`.

In [ ]:
# Define your metric
# Option A: Logit difference (if you have a clear correct vs incorrect)
def metric_logit_diff(logits):
    correct_id = model.to_single_token(expected_token)
    # incorrect_id = model.to_single_token("...")  # fill in
    # return (logits[0, -1, correct_id] - logits[0, -1, incorrect_id]).item()
    pass

# Option B: Logit vs mean (if no clear incorrect token)
def metric_logit_vs_mean(logits):
    correct_id = model.to_single_token(expected_token)
    return (logits[0, -1, correct_id] - logits[0, -1].mean()).item()

# Also define a corrupted prompt that breaks the behavior
corrupted_prompt = "..."  # fill in — same structure but behavior should fail

# Test your metric
with torch.no_grad():
    clean_logits = model(clean_prompt)
    corrupted_logits = model(corrupted_prompt)
    print(f"Clean metric:     {metric_logit_vs_mean(clean_logits):.4f}")
    print(f"Corrupted metric: {metric_logit_vs_mean(corrupted_logits):.4f}")
    print(f"Gap: {metric_logit_vs_mean(clean_logits) - metric_logit_vs_mean(corrupted_logits):.4f}")

## Step 2: Localize — Which Layers Matter?

Activation patching at the layer level. Patch each layer's residual stream from the clean run into the corrupted run. This tells you *where* in the network the behavior lives.

**Note:** Patching `hook_resid_post` at layer L replaces the *cumulative* residual stream through layer L (i.e., the sum of the embedding plus all layers 0 through L), not the contribution of layer L alone. A large recovery at layer L therefore means the information is present by layer L, not necessarily that layer L itself is the sole contributor.

In [ ]:
# Layer-level activation patching
with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(clean_prompt)
    corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_prompt)

    clean_metric = metric_logit_vs_mean(clean_logits)
    corrupted_metric = metric_logit_vs_mean(corrupted_logits)

    layer_results = []
    for layer in range(model.cfg.n_layers):
        def patch_layer(activation, hook, l=layer):
            activation[0] = clean_cache[f"blocks.{l}.hook_resid_post"][0]
            return activation
        patched_logits = model.run_with_hooks(
            corrupted_prompt,
            fwd_hooks=[(f"blocks.{layer}.hook_resid_post", patch_layer)]
        )
        result = metric_logit_vs_mean(patched_logits)
        normalized = (result - corrupted_metric) / (clean_metric - corrupted_metric)
        layer_results.append(normalized)

plt.figure(figsize=(12, 5))
plt.bar(range(model.cfg.n_layers), layer_results,
        color=['#e74c3c' if r > 0.1 else '#95a5a6' for r in layer_results])
plt.xlabel("Layer")
plt.ylabel("Fraction of metric recovered")
plt.title(f"Layer Patching: {BEHAVIOR_NAME}")
plt.axhline(y=0, color='black', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

important_layers = [i for i, r in enumerate(layer_results) if r > 0.1]
print(f"Important layers (>10% recovery): {important_layers}")

## Step 3: Zoom In — Which Heads Matter?

Now patch individual attention heads to find the key players.

In [ ]:
# Head-level activation patching
head_results = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)

with torch.no_grad():
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            def patch_head(activation, hook, l=layer, h=head):
                activation[0, :, h, :] = clean_cache[f"blocks.{l}.attn.hook_result"][0, :, h, :]
                return activation
            patched_logits = model.run_with_hooks(
                corrupted_prompt,
                fwd_hooks=[(f"blocks.{layer}.attn.hook_result", patch_head)]
            )
            result = metric_logit_vs_mean(patched_logits)
            head_results[layer, head] = (result - corrupted_metric) / (clean_metric - corrupted_metric)

plt.figure(figsize=(14, 8))
plt.imshow(head_results.numpy(), cmap="RdBu", vmin=-0.3, vmax=0.3, aspect="auto")
plt.colorbar(label="Fraction of metric recovered")
plt.xlabel("Head")
plt.ylabel("Layer")
plt.title(f"Head Patching: {BEHAVIOR_NAME}")
plt.tight_layout()
plt.show()

# Print top positive and negative heads
flat = [(head_results[l, h].item(), l, h)
        for l in range(model.cfg.n_layers)
        for h in range(model.cfg.n_heads)]
flat.sort(reverse=True)
print("Top positive heads (help the behavior):")
for val, l, h in flat[:5]:
    print(f"  L{l}H{h}: {val:.3f}")
print("Top negative heads (hurt when restored — may be inhibitory):")
for val, l, h in flat[-3:]:
    print(f"  L{l}H{h}: {val:.3f}")

## Step 4: Understand — What Are These Heads Doing?

For each important head, inspect:
1. **Attention patterns** — what tokens does it attend to?
2. **OV circuit** — what information does it move?
3. **Direct logit attribution** — what tokens does it promote?

In [ ]:
# Inspect the top heads you found
# Fill in the head you want to analyze:
LAYER, HEAD = 0, 0  # CHANGE THIS to your top head

with torch.no_grad():
    # 1. Attention pattern
    tokens = [model.tokenizer.decode(t) for t in model.to_tokens(clean_prompt)[0]]
    attn = clean_cache[f"blocks.{LAYER}.attn.hook_pattern"][0, HEAD].detach().cpu()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Attention heatmap
    im = axes[0].imshow(attn.numpy(), cmap="Blues")
    axes[0].set_xticks(range(len(tokens)))
    axes[0].set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
    axes[0].set_yticks(range(len(tokens)))
    axes[0].set_yticklabels(tokens, fontsize=8)
    axes[0].set_title(f"Attention Pattern: L{LAYER}H{HEAD}")
    plt.colorbar(im, ax=axes[0])

    # What the last position attends to
    axes[1].barh(range(len(tokens)), attn[-1].numpy())
    axes[1].set_yticks(range(len(tokens)))
    axes[1].set_yticklabels(tokens, fontsize=8)
    axes[1].set_title(f"L{LAYER}H{HEAD}: Last position attends to...")
    axes[1].set_xlabel("Attention weight")

    plt.tight_layout()
    plt.show()

    # 2. Direct logit attribution — what does this head promote?
    head_output = clean_cache[f"blocks.{LAYER}.attn.hook_result"][0, -1, HEAD]
    # NOTE: This is an approximation that omits the final layer norm (ln_final).
    # For more accurate direct logit attribution, one would need to apply
    # model.ln_final to the head_output before projecting through W_U.
    logit_contrib = head_output @ model.W_U
    top_promoted = logit_contrib.topk(10)
    print(f"\nL{LAYER}H{HEAD} promotes these tokens at the final position:")
    for i in range(10):
        tok = model.to_string([top_promoted.indices[i].item()])
        print(f"  {tok!r}: {top_promoted.values[i].item():.3f}")

## Step 5: Write It Up

The most important step. Fill in the template below with your findings. Be honest about what you don't understand — that's where the interesting research questions are.

## My Circuit Analysis: [BEHAVIOR NAME]

### The Behavior
*What does the model do? What prompt did you use? How reliable is it?*

YOUR ANSWER HERE

### The Metric
*How did you measure the behavior? What was the clean vs corrupted gap?*

YOUR ANSWER HERE

### The Circuit
*Which components matter? Draw a rough diagram:*

```
[input tokens] → ??? → [Layer ?] Head ?.? (does what?) → ??? → [output token]
```

### Evidence
*What specific evidence supports your circuit description?*
- Activation patching showed...
- Attention patterns revealed...
- The OV circuit / direct logit attribution showed...

### What I Don't Understand
*Be honest. What parts of the behavior aren't explained by your circuit?*

YOUR ANSWER HERE

### If I Had More Time
*What would you investigate next?*

YOUR ANSWER HERE

---
## Going Further

Congratulations — you just did real interpretability research. Here's how to level up:

- **Reproduce a published circuit.** Try the greater-than circuit (Hanna et al. 2023) or the gender bias circuit.
- **Post your findings.** Write it up on the Alignment Forum or LessWrong. The community is hungry for reproductions and novel circuit analyses.
- **Scale up.** Try the same analysis on GPT-2 medium or Pythia models. Does the circuit transfer?
- **Contribute to open source.** TransformerLens, SAELens, and Neuronpedia all welcome contributors.

The field is young enough that a single well-done circuit analysis can be a meaningful contribution.